In [1]:
import numpy as np 
import soundfile as sf
import librosa
import pandas as pd 
from pathlib import Path
import pickle 
import IPython.display as ipd
import sys 
sys.path.append('../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 
import soxr

# Make stimuli for 2024 mono cue duration word recognition experiment

## Wanted Conditions:

### SNRs:
- 0 dB SNR
### Distractor conditions:
- 1-talker
### Cue durations in seconds (informed by model experiment results)
- 0.1 
- 0.25
- 0.5
- 1
- 2 


For pilot, use stimuli from 1-talker 0 dB condition:    
`/om/user/imgriff/datasets/human_word_rec_SWC_2023/sounds/condition_33`

New audio will be created using the pre-selected cue and mixture files in this manifest. Cues will be cut to fit the duration for each condition, then concatenated with the existing mixture. 


Stim will be moved to `/om/user/imgriff/datasets/human_cue_duration_SWC_2024`

In [2]:
# manifest = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_experiment_v00/cue_and_target_manifest.pdpkl')

In [3]:
# talker_gender_dict = manifest_all_words.groupby('client_id').gender.unique().to_dict()
# talker_gender_dict = {talker:gender[0] for talker,gender in talker_gender_dict.items()}

In [4]:
stim_manifest = pd.read_pickle('/om/user/imgriff/datasets/human_word_rec_SWC_2023/sounds/condition_33/manifest.pdpkl')


In [8]:
stim_manifest['sex_cond'] = stim_manifest['target_gender'] == stim_manifest['distractor_gender']
stim_manifest['sex_cond'] = stim_manifest['sex_cond'].apply(lambda x: "Same" if x else "Different")

In [10]:
stim_manifest['sex_cond'].value_counts()

Same         190
Different    170
Name: sex_cond, dtype: int64

In [4]:
stim_out_path = Path('/om/user/imgriff/datasets/human_cue_duration_SWC_2024')
stim_out_path.mkdir(exist_ok=True, parents=True)
stim_manifest = pd.read_pickle('/om/user/imgriff/datasets/human_word_rec_SWC_2023/sounds/condition_33/manifest.pdpkl')
stim_manifest.shape

(360, 18)

In [13]:
stim_manifest.to_pickle(stim_out_path / 'manifest.pdpkl') # copy to new location

In [22]:
stim_manifest.columns

Index(['target_sr', 'experiment_key_target_word_ix', 'cue_sr', 'target_fn',
       'cue_fn', 'word', 'word_int', 'condition', 'snr', 'src_ix', 'client_id',
       'target_gender', 'target_f0', 'distractor_fn', 'distractor_f0',
       'distractor_gender', 'distractor_word', 'mixture_fn'],
      dtype='object')

In [6]:
stim_manifest.head()

,target_sr,experiment_key_target_word_ix,cue_sr,target_fn,cue_fn,word,word_int,condition,snr,src_ix,client_id,target_gender,target_f0,distractor_fn,distractor_f0,distractor_gender,distractor_word,mixture_fn
0,44100,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,above,1,1-talker,0,703492,flyingtoaster,female,163.753088,/om/user/imgriff/datasets/spatial_audio_pipeli...,145.594463,male,were,/om/user/imgriff/datasets/human_word_rec_SWC_2...
1,44100,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,according,3,1-talker,0,244898,gorillawarfare,female,229.612578,/om/user/imgriff/datasets/spatial_audio_pipeli...,87.510941,male,board,/om/user/imgriff/datasets/human_word_rec_SWC_2...
2,44100,2,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,across,4,1-talker,0,126322,popularoutcast,female,180.108941,/om/user/imgriff/datasets/spatial_audio_pipeli...,133.414830,male,waiting,/om/user/imgriff/datasets/human_word_rec_SWC_2...
3,44100,3,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,action,5,1-talker,0,937425,preslethe,male,114.925214,/om/user/imgriff/datasets/spatial_audio_pipeli...,183.153323,male,military,/om/user/imgriff/datasets/human_word_rec_SWC_2...
4,44100,4,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,activities,7,1-talker,0,940458,vaslittlecrow,female,207.809580,/om/user/imgriff/datasets/spatial_audio_pipeli...,125.965637,male,worked,/om/user/imgriff/datasets/human_word_rec_SWC_2...


In [21]:
ipd.display(ipd.Audio(stim_manifest.iloc[0].cue_fn, normalize=False))
ipd.display(ipd.Audio(stim_manifest.iloc[0].mixture_fn, normalize=False))

In [9]:
# Make condition dict 
import itertools 
import pickle 
import numpy as np 


cue_durations = [0.1, 0.25, 0.5, 1, 2]


cond_ix_dict = {ix:dur for ix, dur in enumerate(cue_durations)}

out_name = "human_pilot_cue_duration_cond_map.pkl"
# write condition dict as pickle 
with open(out_name, 'wb') as f:
    pickle.dump(cond_ix_dict, f)


In [11]:
cond_ix_dict

{0: 0.1, 1: 0.25, 2: 0.5, 3: 1, 4: 2}

In [12]:
# Make word key 

target_words = np.sort(stim_manifest['word'].unique())
#enumerate words as dict 
target_word_dict = dict(enumerate(target_words))
# save as pickle
out_name = "human_pilot_cue_duration_word_key.pkl"
with open(out_name, 'wb') as f:
    pickle.dump(target_word_dict, f)


In [44]:
(target_words ==  stim_manifest['word'].values).all() # already in order 

True

In [14]:
# Crop waveforms to given duration keeping center segment 
def crop_wav(wav, sr, duration):
    n_samples = int(duration*sr)
    start_idx = int((len(wav) - n_samples)/2)
    end_idx = int(start_idx + n_samples)
    return wav[start_idx:end_idx]

def get_excerpt(dfi, dur=3.0, sr=50000, pad_with_context=True, jitter_fraction=0):
    """
    This function loads an audio file and excerpts a clip with the specified
    duration. Target durations that exceed clip boundaries are handled with
    zero-padding (applied to all signals but sliced away when not needed).
    This function also handles resampling (via soxr) and re-scaling.
    """
    jitter_in_s = 0
    jitter_via_zero_padding = True
    if dfi.clip_dur_in_s > dur:
        # Take a random segment if clip duration is longer than excerpt
        clip_start_in_s = np.random.uniform(
            low=dfi.clip_start_in_s,
            high=dfi.clip_start_in_s + dfi.clip_dur_in_s - dur,
            size=None)
        clip_end_in_s = clip_start_in_s + dur
        jitter_via_zero_padding = False
    else:
        # Temporally jitter clip by extending either start or end time
        jitter_in_s = np.random.uniform(
            low=-dfi.clip_dur_in_s * jitter_fraction,
            high=dfi.clip_dur_in_s * jitter_fraction,
            size=None)
        if pad_with_context:
            # If using context, adjust clip start and end times to account for jitter and context
            if jitter_in_s > 0:
                clip_start_in_s = dfi.clip_start_in_s - (2 * np.abs(jitter_in_s))
                clip_end_in_s = dfi.clip_end_in_s
            else:
                clip_start_in_s = dfi.clip_start_in_s
                clip_end_in_s = dfi.clip_end_in_s + (2 * np.abs(jitter_in_s))
            clip_dur_in_s = clip_end_in_s - clip_start_in_s
            jitter_via_zero_padding = False
            context_pad_in_s = (dur - clip_dur_in_s) / 2
        else:
            clip_start_in_s = dfi.clip_start_in_s
            clip_end_in_s = dfi.clip_end_in_s
            context_pad_in_s = 0
        clip_start_in_s = clip_start_in_s - context_pad_in_s
        clip_end_in_s = clip_end_in_s + context_pad_in_s
    clip_dur_in_s = clip_end_in_s - clip_start_in_s
    # Load audio, pad, slice with indexes that account for padding
    load_full_file = True
    if (clip_start_in_s >= 0) and (clip_end_in_s < dfi.total_file_duration_in_s):
        # Attempt to read only the specified excerpt
        myfile = sf.SoundFile(dfi.src_fn)
        if myfile.seekable():
            src_sr = myfile.samplerate
            frame_start = int(np.round(clip_start_in_s * src_sr))
            frames = int(np.round(clip_dur_in_s * src_sr))
            myfile.seek(frame_start)
            y = myfile.read(frames, always_2d=True)
            y = np.mean(y, axis=1)
            load_full_file = False
    if load_full_file:
        # If impossible, read full audio file
        y, src_sr = sf.read(dfi.src_fn, always_2d=True)
        y = np.mean(y, axis=1)
        frame_start = int(np.round(clip_start_in_s * src_sr))
        frames = int(np.round(clip_dur_in_s * src_sr))
        if frame_start < 0:
            y = np.pad(y, [-frame_start, 0])
            frame_start = 0
        if frame_start + frames > len(y):
            y = np.pad(y, [0, frame_start + frames - len(y)])
        y = y[frame_start : frame_start + frames]
    # Resample from src_sr to sr
    y = soxr.resample(y, src_sr, sr).astype(np.float32)
    # If not yet jittered, apply jitter at end via asymmetric zero-padding
    if jitter_via_zero_padding:
        jitter_pad_width = int(np.round(2 * np.abs(jitter_in_s) * sr))
        if jitter_in_s > 0:
            y = np.pad(y, [jitter_pad_width, 0]).astype(np.float32)
        else:
            y = np.pad(y, [0, jitter_pad_width]).astype(np.float32)
    # Zero-pad or trim to length (fixes off by one errors)
    y = util_audio.pad_or_trim_to_len(y, int(dur * sr))
    y = np.nan_to_num(y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return y

def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factor

def rms_normalize_db(wav, dBSPL, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    new_rms = 20e-6 * np.power(10, dBSPL/20)
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor



In [41]:
from tqdm.auto import tqdm

SR = 44100
# timing in seconds 
# trial_dur = 4.5
signal_dur = 2
isi = 0.5
win_dur = 10 # ms 
mixture_onset = int(2 * SR) # in frames; mixture will include ISI and signal 
# make dir 
stim_out_path = Path('/om/user/imgriff/datasets/human_cue_duration_SWC_2024/sounds')

stim_out_path.mkdir(parents=True, exist_ok=True)
records = []

# for cond_ix, (condition, snr) in tqdm(cond_ix_dict.items()):

# target word dict is defined above 

cond_records = []

for cond_ix, cue_dur in tqdm(cond_ix_dict.items()):

    trial_dur = cue_dur + signal_dur + isi

    for ix, word in tqdm(target_word_dict.items()):
        row = stim_manifest[stim_manifest.word == word].iloc[0]
        trial_signal = np.zeros((int(SR * trial_dur)),dtype=np.float32)

        mixture, target_sr = librosa.load(row.mixture_fn, sr=SR, offset=2) 
        mixture = util_audio.ramp_hann(mixture, win_dur, samplerate=SR)

        cue_wav, cue_sr = librosa.load(row.cue_fn, sr=SR)
        # center crop cue_wav to desired duration
        cue_wav = crop_wav(cue_wav, SR, cue_dur)
        cue_wav = util_audio.ramp_hann(cue_wav, win_dur, samplerate=SR)


        mixture, mixture_scale_factor = rms_normalize_db(mixture, 60)
        # set cue to same level as target post scaling 
        cue_wav = util_audio.set_dBSPL(cue_wav, 60)
        cue_wav *= mixture_scale_factor

        # add cue to trial signal
        trial_signal[:len(cue_wav)] += cue_wav    
        trial_signal[len(cue_wav):] += mixture
        trial_signal = util_audio.set_dBSPL(trial_signal, 60)

        # save trial signal
        # f string with digint padded to 3 places
        out_name = stim_out_path / f"condition_{cond_ix}"/ f'{ix:03}.wav'
        # make out name directory if it doesn't exist
        out_name.parent.mkdir(parents=True, exist_ok=True)
        sf.write(out_name, trial_signal, samplerate=SR)
        # if ix == 10:
    #     break 

# break


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

In [33]:
trial_signal.shape[0]/SR

2.6

## Save word key maps  for stimuli


In [48]:
## Make word Key .js from pickle of word dict
import json
import pickle 

word_key_dict = pickle.load(open("human_pilot_cue_duration_word_key.pkl", 'rb'))

words = list(word_key_dict.values())
out_dir = Path("/mindhive/mcdermott/www/imgriff/msjspsych/cocktail_party_cue_duration_pilot/stim/")
# save list of experiment words as json 
out_name = "key.js"
with open( out_dir / out_name, 'w') as f:
    json.dump({"key": words}, f)
# words

In [7]:
## Make sure files actually saved 
from pathlib import Path 
import librosa

word_conds = np.arange(360)
condition_ixs = np.arange(5)
stim_dir = Path('/mindhive/mcdermott/www/imgriff/msjspsych/cocktail_party_cue_duration/stim/')

all_paths = []
bad_paths = []
for word in word_conds:
    for cond in condition_ixs:
        stim = stim_dir / f'condition_{cond}/{word:03}.wav'
        all_paths.append(stim)
        try:
            librosa.load(stim, sr=44_100)
        except Exception as e:
            bad_paths.append(stim)
            print(e)
            continue